### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [215]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [216]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [217]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [218]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'), data_home='./data')
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'), data_home='./data')

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [219]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [220]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [221]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [222]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [223]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [224]:
#tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [225]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [226]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [227]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [228]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [229]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [230]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [231]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  6385,  1149, 11238], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [232]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [233]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [234]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [235]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [236]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [237]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


### 1. Vectorizar documentos
Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

In [238]:
# Toma 5 documentos al azar (índices)

# X_train: Matriz generada por TfidfVectorizer a partir de los documentos de entrenamiento
# Filas: Documentos
# Columnas: Palabras del vocabulario

np.random.seed(42)
selected_indexes = np.random.choice(X_train.shape[0], size=5, replace=False)

print(f"Índices seleccionados: {selected_indexes}")

Índices seleccionados: [7492 3546 5582 4793 3813]


In [239]:
# Función para obtener los índices de los documentos más similares a un documento de consulta dado su índice.

def get_similarity_indexes(documet_index, document_list, top_n=5):

    # Calcula la similitud coseno del documento de consulta con todos los documentos en la matriz.
    sim_scores = cosine_similarity(document_list[documet_index], document_list)[0]
    
    # Obtiene los top_n + 1 índices de los documentos ordenados por similitud (de mayor a menor).
    top_indexes = sim_scores.argsort()[-(top_n + 1):][::-1]
    
    # Elimina el índice del documento de consulta (el primer índice).
    top_indexes = top_indexes[1:]
    
    # Extrae los puntajes de similitud correspondientes a los índices finales.
    top_scores = sim_scores[top_indexes]
    
    return top_indexes, top_scores

In [240]:
# Obtiene los índices de los documentos más similares para cada uno de los documentos seleccionados al azar.

import pandas as pd
import textwrap
from IPython.display import display # Para que se vea bien en Jupyter/Colab

# Ajusta Pandas para que corte el texto por la mitad
pd.set_option('display.max_colwidth', 200)

for i, idx in enumerate(selected_indexes):

    # Datos del documento seleccionado
    selected_label = newsgroups_train.target_names[newsgroups_train.target[idx]]
    selected_text_clean = newsgroups_train.data[idx].replace('\n', ' ')[:600]
    selected_formated_text = textwrap.fill(selected_text_clean, width=150)
    
    print(f"\n{'='*150}")
    print(f"SELECCIONADO #{i+1} | Índice: {idx} | Etiqueta: {selected_label}")
    print(f"Fragmento Texto:")
    print(f"{selected_formated_text}")
    print(f"{'='*150}")
    
    # Obtiene índices y scores de mayor similitud para el documento seleccionado
    similar_indexes, similar_scores = get_similarity_indexes(idx, X_train, top_n=5)
    
    # Crea lista vacía para guardar los resultados
    results_list = []
    
    for j, (sim_idx, sim_score) in enumerate(zip(similar_indexes, similar_scores), start=1):
        sim_label = newsgroups_train.target_names[newsgroups_train.target[sim_idx]]
        sim_text_clean = newsgroups_train.data[sim_idx].replace('\n', ' ')[:400] 
        
        # Guarda cada fila como diccionario
        results_list.append({
            'Orden': j,
            'Índice': sim_idx,
            'Similitud': round(sim_score, 4),
            'Etiqueta': sim_label,
            'Fragmento Texto': f"{sim_text_clean}..."
        })
    
    # Convertimos la lista de diccionarios en un DataFrame de Pandas
    df_results = pd.DataFrame(results_list)
    
    # Mostramos la tabla sin el índice por defecto de Pandas para que quede más limpia
    display(df_results.style.hide(axis="index"))


SELECCIONADO #1 | Índice: 7492 | Etiqueta: comp.sys.mac.hardware
Fragmento Texto:
Could someone please post any info on these systems.  Thanks. BoB --  ----------------------------------------------------------------------  Robert
Novitskey | "Pursuing women is similar to banging one's head rrn@po.cwru.edu  |  against a wall...with less opportunity for reward"


Orden,Índice,Similitud,Etiqueta,Fragmento Texto
1,10935,0.666500,comp.sys.mac.hardware,"Hey everybody: I want to buy a mac and I want to get a good price...who doesn't? So, could anyone out there who has found a really good deal on a Centris 650 send me the price. I don't want to know where, unless it is mail order or areound cleveland, Ohio. Also, should I buy now or wait for the Power PC. Thanks. BoB reply via post or e-mail at rrn@po.cwru.edu -- --------------------------..."
2,7258,0.347600,comp.sys.ibm.pc.hardware,Hay all: Has anyone out there heard of any performance stats on the fabled p24t. I was wondering what it's performance compared to the 486/66 and/or pentium would be. Any info would be helpful. Later BoB -- Robert Novitskey | rrn@po.cwru.edu | (216)754-2134 | CWRU Cleve. Ohio ---------------------------------------------------------------------- COMPUTER ENGINEER AND C PROGRAMMER | NOW S...
3,4971,0.179900,comp.sys.mac.hardware,Could someone please send instructions for installing simms and vram to jmk13@po.cwru.edu? He's just gotten his 700 and wants to drop in some extra simms and vram that he has for it....
4,4303,0.154700,misc.forsale,For sale: Roland D-50: $700 or best offer. Excellent condition. Includes over 1000 patches on disk (In cakewalk sysex format) Buyer must pay COD shipping. Please e-mail responses to: gms2@po.cwru.edu Thanks. George ...
5,645,0.141400,comp.sys.mac.hardware,"I'd appreciate it greatly if someone could E-mail me the following: (if you only know one, that's fine) 1) Specs for the 68040 (esp. how it compares to the Pentium) 2) Specs for the 68060 with estimated cost, release date, etc... I'm interested in speeds, systems it can run (Windows NT, RISC, or whatever), costs, bus info, register info. All the technical info. I am hoping that the 68040 can wi..."



SELECCIONADO #2 | Índice: 3546 | Etiqueta: comp.os.ms-windows.misc
Fragmento Texto:
       Don't bother if you have CPBackup or Fastback.  They all offer options  not available in the stripped-down MS version (FROM CPS!).  Examples -
no  proprietary format (to save space), probably no direct DMA access, and no  tape drive!


Orden,Índice,Similitud,Etiqueta,Fragmento Texto
1,5665,0.204000,comp.sys.ibm.pc.hardware,"By initiating a DMA xfer. :) Seriously, busmastering adapter have their own DMA ability, they don't use the motherboards on-board DMA(which is *MUCH* slower). ISA has no bus arbitration, so if two busmastering cards in 1 ISA system try to do DMA xfers on the same DMA channel the system will lock or crash.(I forget) Their are 8 DMA channels in an ISA system. 0-7. 0-3 are 8-bit & 4-7 are 16-bi..."
2,2011,0.192400,comp.sys.ibm.pc.hardware,"IDE also uses DMA techniques. I believe floppy controller also uses DMA, and most A/D boards also use DMA. DMA is no big deal, and has nothing to do directly with SCSI. You can thank your software for that. If DOS had a few more brains, it could format floppies etc. while you were doing something else. The hardware will support it, but DOS (at least) won't. Again, this has nothing to d..."
3,8643,0.172400,comp.sys.ibm.pc.hardware,"There would be no problems as long as the OS didn't set up a DMA transfer to an area above the 16 mb area (the DMA controller probably can't be programmed that way anyways, so there probably isin't a problem with this)..."
4,1546,0.170900,comp.sys.ibm.pc.hardware,"Here's a document that I wrote some time back. It's slightly out-of-date, now that DOS 6 has been released, but much of it is still useful. -- Darryl Okahata Internet: darrylo@sr.hp.com DISCLAIMER: this message is the author's personal opinion and does not constitute the support, opinion or policy of Hewlett-Packard or of the little green men that have been following him all day. ..."
5,8765,0.161600,comp.sys.ibm.pc.hardware,"The floppy is served by DMA on the motherboard, and original DMA-controller can't reach more than the first 16MB (The address-space of the ISA-bus) joerg ..."



SELECCIONADO #3 | Índice: 5582 | Etiqueta: misc.forsale
Fragmento Texto:
5.25" Internal Low density disk drive.  Monochrome monitor  8088 motherboard, built in parallel and serial ports, built in mono and color output,
7Mhz.  Libertarian, atheist, semi-anarchal Techno-Rat.


Orden,Índice,Similitud,Etiqueta,Fragmento Texto
1,5510,0.462200,misc.forsale,"I am looking for a 286 motherboard, preferable 12 or 16, 640k or 1 meg RAM. I am also looking for a VGA card. Am willing to trade 1200 external, 5.25"" LD Drive, 8088 motherboard, monochrome monitor, Game Boy, in some combination for the above. Libertarian, atheist, semi-anarchal Techno-Rat...."
2,4922,0.299900,misc.forsale,"For sale: Nintendo Game Boy, Tetris, Castlevania Adventure, All-Star Challenge, Nemesis, Play-Action football, link cable. Make me an offer. Libertarian, atheist, semi-anarchal Techno-Rat...."
3,4347,0.274000,comp.graphics,"It's really not that hard to do. There are books out there which explain everything, and the basic 3D functions, translation, rotation, shading, and hidden line removal are pretty easy. I wrote a program in a few weeks witht he help of a book, and would be happy to give you my source. Also, Quickdraw has a lot of 3D functions built in, and Think pascal can access them, and I would expect that..."
4,8057,0.207600,misc.forsale,"Hello, I have a motherboard and a case for sale as a package. Both of them came from a CompuAdd computer I bought last August and am presently upgrading. Here are the specs-- Motherboard ----------- Cyrix 486SL 25 MHz microprocessor Chips and Technology chipset (SCATsx V2.3.6 SLSLC) 8 SIMM banks for a maximum of 32 Megs of RAM BUILT-IN Floppy and Hard Drive Controllers BUILT-IN ports--1 Parall..."
5,4028,0.168500,comp.graphics,"Hello, and thank you for reading this request. I have a Mpeg viewer for x-windows and it did not run because I was running it on a monochrome monitor. I need the mono-driver for mpeg_play. ..."



SELECCIONADO #4 | Índice: 4793 | Etiqueta: talk.politics.guns
Fragmento Texto:
Hi,  In Canada, any gun that enters a National Park must be sealed (I think it's a small metal tag that's placed over the trigger).  The net result of
this is that you _can't_ use a gun to protect yourself from bears (or psychos) in the National Parks.  Instead, one has to be sensitive to the dangers
and annoyances of hiking in bear country, and take the appropriate precautions.  I think this policy makes the users of the National Parks feel a
little closer to Nature, that they are a part of Nature and, as such, have to deal with nature on it's own terms.


Orden,Índice,Similitud,Etiqueta,Fragmento Texto
1,6894,0.236400,talk.politics.guns,"Here is a press release from the White House. President Clinton's Remarks On Waco With Q/A To: National Desk Contact: White House Office of the Press Secretary, 202-456-2100 WASHINGTON, April 20 -- Following are remarks by President Clinton in a question and answer session with the press: 1:36 P.M. EDT THE PRESIDENT: On February the 28th, four federal agents were killed in the lin..."
2,5856,0.236300,sci.crypt,"Thanks for posting this and making it available. This post will be LONG, I will comment on most of it, and am reluctantly leaving all of the original in place to provide context. Please note that an alt. group has been set up for the Clipper stuff. ^^^^^^^^^ Hum, AT&T, VLSI and Mykotronx are 'industry'? Wonder what happened to IBM, this shoul..."
3,4271,0.232800,talk.politics.misc,"THE WHITE HOUSE Office of the Press Secretary ______________________________________________________________ For Immediate Release April 13, 1993 REMARKS BY THE PRESIDENT, SECRETARY OF EDUCATION RICHARD RILEY AND SECRETARY OF LABOR ROBERT REICH IN GOALS 2000 SATEL..."
4,3141,0.229500,talk.politics.guns,"Mr. Parsli, I have to take exception at this. There are verifiable, previous *examples* of levels of U.S. governments abusing gun-control restrictions. I don't think it is paranoid to worry that what has been abused in the recent past might be abused in thye future. After so many times of getting burned any sane person will stop putting his hand on the stove. I'd love to. But ..."
5,10836,0.229100,alt.atheism,"Archive-name: atheism/faq Alt-atheism-archive-name: faq Last-modified: 5 April 1993 Version: 1.1 Alt.Atheism Frequently-Asked Questions This file contains responses to articles which occur repeatedly in alt.atheism. Points covered here are ones which are not covered in the ""Introduction to Atheism""; you are advised to read that article as well before posting. These answers ..."



SELECCIONADO #5 | Índice: 3813 | Etiqueta: rec.sport.hockey
Fragmento Texto:
 Doesn't it also have the Statue of Liberty on it or is that Richter's Mask?  The back actually has a Bee followed by a Z to represent the Beezer. It
also has something that looks like the three interconnecting circles from the Led Zepplin 4 album cover. Is that what it is supposed to be? and if it
is does anybody know why he would put it there? Ali?   John "The official Language of Golf is Profanity"


Orden,Índice,Similitud,Etiqueta,Fragmento Texto
1,10836,0.251400,alt.atheism,"Archive-name: atheism/faq Alt-atheism-archive-name: faq Last-modified: 5 April 1993 Version: 1.1 Alt.Atheism Frequently-Asked Questions This file contains responses to articles which occur repeatedly in alt.atheism. Points covered here are ones which are not covered in the ""Introduction to Atheism""; you are advised to read that article as well before posting. These answers ..."
2,759,0.248000,soc.religion.christian,"Oh contrer mon captitan! There is a way. Certainly it is not by human reason. Certainly it is not by human experience. (and yet it is both!) To paraphrase Sartre, the particular is absurd unless it has an infinite reference point. It is only because of God's own revelation that we can be absolute about a thing. Your logic comes to fruition in relativism. Ah, now it is clear. Ludwig was ..."
3,913,0.241000,alt.atheism,"The recent rise of nostalgia in this group, combined with the incredible level of utter bullshit, has prompted me to comb through my archives and pull out some of ""The Best of Alt.Atheism"" for your reading pleasure. I'll post a couple of these a day unless group concensus demands that I stop, or I run out of good material. I haven't been particularly careful in the past about saving ..."
4,5826,0.240900,soc.religion.christian,"A listmember (D Andrew Killie, I think) wrote, in response to the suggestion that genocide may sometimes be the will of God: > Any God who works that way is indescribably evil, > and unworthy of my worship or faith. Nobuya ""Higgy"" Higashiyama replied (as, in substance, did others): > Where is your source of moral standards by which you judge God's > behavior? It is often argued that we hav..."
5,5856,0.232900,sci.crypt,"Thanks for posting this and making it available. This post will be LONG, I will comment on most of it, and am reluctantly leaving all of the original in place to provide context. Please note that an alt. group has been set up for the Clipper stuff. ^^^^^^^^^ Hum, AT&T, VLSI and Mykotronx are 'industry'? Wonder what happened to IBM, this shoul..."


#### Análisis de resultados

1. Documento #1: El documento seleccionado tiene como etiqueta comp.sys.mac.hardware y trata sobre un usuario solicitando información sobre algún tipo de sistema informático. Los textos de los documentos con mayor similitud obtenidos tienen etiquetas tambien relacionadas con hardware informatico o compra / venta del mismo. Todos tratan sobre temas relacionados con el seleccionado: compras / ventas de equipamiento, ayuda para instalacion de hardware, etc.

2. Documento #2: El documento seleccionado tiene como etiqueta comp.os.ms-windows.misc y trata sobre una discucción sobre como se realizanban las copias de seguridad en el antiguo windows. Menciona el término DNA (acceso directoa memoria), entiendo un término no muy comun en los documentos por lo que mayormente los reusltados mas similares tratan temas relacionados a eso.  

3. Documento #3: El documento seleccionado tiene como etiqueta cmisc.forsale y trata sobre la venta de productos informaticos. La mayoria de las similitudes tambien son sobre ventas de equipos informáticos con la misma etiqueta y algunas tratan temas de gráficos usando algunos términos similares a los del hardware ofrecido en el documento seleccionado. 

4. Documento #4: El documento seleccionado tiene como etiqueta talk.politics.guns y trata sobre las leyes portacion de armas en canada. Los mas similares seleccionados tambien son de tematicas sobre armas o medidas gubernamentales en general. 

5. Documento #5: El documento seleccionado tiene como etiqueta rec.sport.hockey, si bien esta en un foro de deportes, el documento trata particularmente sobre como esta ilustrado el casco de un arquero de hockey, el arte que tiene. Esto lleva a que los resultados de similitud sean de etiquetas variadas y en si de temas no muy relacionados finalmente. 


### 2. Construir un modelo de clasificación por prototipos (tipo zero-shot)
Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

In [241]:
from sklearn.metrics import classification_report

# Obtiene una matriz de similitud de todos los documentos de test contra todos los documentos de train
sim_matrix = cosine_similarity(X_test, X_train)

# Obtiene un vector con el índice del documento de train más similar para cada documento de test
most_similar_train_indexes = sim_matrix.argmax(axis=1)

# Asigna la etiqueta del documento de train más similar a cada documento de test
y_pred_sim = y_train[most_similar_train_indexes]

print("Reporte de clasificación utilizando la etiqueta del documento de entrenamiento más similar:\n")
print(classification_report(y_test, y_pred_sim, target_names=newsgroups_train.target_names))

Reporte de clasificación utilizando la etiqueta del documento de entrenamiento más similar:

                          precision    recall  f1-score   support

             alt.atheism       0.37      0.51      0.43       319
           comp.graphics       0.54      0.48      0.51       389
 comp.os.ms-windows.misc       0.51      0.46      0.48       394
comp.sys.ibm.pc.hardware       0.52      0.52      0.52       392
   comp.sys.mac.hardware       0.53      0.50      0.52       385
          comp.windows.x       0.70      0.59      0.64       395
            misc.forsale       0.63      0.46      0.53       390
               rec.autos       0.41      0.58      0.48       396
         rec.motorcycles       0.63      0.52      0.57       398
      rec.sport.baseball       0.65      0.54      0.59       397
        rec.sport.hockey       0.75      0.72      0.73       399
               sci.crypt       0.55      0.59      0.57       396
         sci.electronics       0.53      0.33   

#### Análisis de resultados
Con una exactitud de 0.51 y un F1 promedio de 0.50 los resultados del modelo no son buenos, pero establece la base sobre la cual evaluar mejoras de otros modelos

### 3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación
F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

In [242]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, classification_report

# Funcion para optimizar modelos con GridSearchCV
def model_optimizer(model_instance):
    
    # Define un pipeline que incluye el vectorizador TF-IDF y el modelo de clasificación
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english')),
        ('clf', model_instance)
    ])

    # Grilla de parámetros permitidos que  utilizara GridSearchCV para optimizar el modelo. 
    param_grid = {
        'tfidf__sublinear_tf': [True, False],  # Suaviza el peso de palabras repetidas en un mismo texto.
        'tfidf__max_df': [0.5, 0.75, 1.0],     # Ignora palabras que están en más del 50% o 75% de los textos (stop-words).
        'tfidf__min_df': [1, 2, 5],            # Ignora palabras rarísimas o errores de tipeo.
        'clf__alpha': [0.01, 0.1, 0.5, 1.0],   # Suavizado del modelo.
        'clf__fit_prior': [True, False]        # Aprender o no las probabilidades previas de las clases.
    }

    # Crea el GridSearchCV con el pipeline, la grilla de parámetros, validación cruzada de 3 folds, y scoring por F1-score macro.
    grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='f1_macro', n_jobs=-1)

    # Ejecuta el GridSearchCV para encontrar los mejores parámetros del modelo utilizando los datos de entrenamiento.
    grid_search.fit(newsgroups_train.data, newsgroups_train.target)
    
    return grid_search

In [243]:
# Optimiza MultinomialNB
res_multinomial = model_optimizer(MultinomialNB())

# Resultados MultinomialNB
print("\nMODELO: Multinomial Naïve Bayes")
print(f"Mejor F1-Score (Validación): {res_multinomial.best_score_:.4f}")
print("Parámetros que generaron este puntaje:")
for parametro, valor in res_multinomial.best_params_.items():
    print(f"  - {parametro}: {valor}")

# Genera las predicciones 
y_pred_multinomial = res_multinomial.predict(newsgroups_test.data)

# Imprime el reporte
print("\n")
print(classification_report(newsgroups_test.target, y_pred_multinomial, target_names=newsgroups_test.target_names))


MODELO: Multinomial Naïve Bayes
Mejor F1-Score (Validación): 0.7498
Parámetros que generaron este puntaje:
  - clf__alpha: 0.01
  - clf__fit_prior: False
  - tfidf__max_df: 0.5
  - tfidf__min_df: 1
  - tfidf__sublinear_tf: False


                          precision    recall  f1-score   support

             alt.atheism       0.31      0.48      0.38       319
           comp.graphics       0.66      0.71      0.68       389
 comp.os.ms-windows.misc       0.70      0.53      0.60       394
comp.sys.ibm.pc.hardware       0.61      0.69      0.65       392
   comp.sys.mac.hardware       0.72      0.70      0.71       385
          comp.windows.x       0.80      0.74      0.77       395
            misc.forsale       0.79      0.72      0.76       390
               rec.autos       0.75      0.72      0.74       396
         rec.motorcycles       0.76      0.71      0.73       398
      rec.sport.baseball       0.94      0.81      0.87       397
        rec.sport.hockey       0.92      

In [244]:
# Optimiza ComplementNB
res_complement = model_optimizer(ComplementNB())

# Resultados ComplementNB
print("\nMODELO: Complement Naïve Bayes")
print(f"Mejor F1-Score (Validación): {res_complement.best_score_:.4f}")
print("Parámetros que generaron este puntaje:")
for parametro, valor in res_complement.best_params_.items():
    print(f"  - {parametro}: {valor}")

# Genera las predicciones
y_pred_complement = res_complement.predict(newsgroups_test.data)

# Imprimimos el reporte
print("\n")
print(classification_report(newsgroups_test.target, y_pred_complement, target_names=newsgroups_test.target_names))


MODELO: Complement Naïve Bayes
Mejor F1-Score (Validación): 0.7558
Parámetros que generaron este puntaje:
  - clf__alpha: 0.5
  - clf__fit_prior: True
  - tfidf__max_df: 0.5
  - tfidf__min_df: 1
  - tfidf__sublinear_tf: True


                          precision    recall  f1-score   support

             alt.atheism       0.32      0.43      0.37       319
           comp.graphics       0.73      0.71      0.72       389
 comp.os.ms-windows.misc       0.74      0.56      0.64       394
comp.sys.ibm.pc.hardware       0.64      0.71      0.67       392
   comp.sys.mac.hardware       0.77      0.74      0.75       385
          comp.windows.x       0.81      0.80      0.80       395
            misc.forsale       0.75      0.73      0.74       390
               rec.autos       0.81      0.75      0.78       396
         rec.motorcycles       0.84      0.76      0.80       398
      rec.sport.baseball       0.92      0.84      0.88       397
        rec.sport.hockey       0.87      0.94

#### Análisis de resultados

Ambos modelos obtienen resultados similares: F1 Macro Avg = 0.69 para MultinomialNB y F1 macro Avg = 070 para ComplementNB. Ambos mejoran el prototipo usado de base com F1 Macro Avg = 0.50. Que ambos modelos tengan resultados similares se explica principalmente porque el dataset no presenta grandes desbalances GridSearch establecio en false el parametro fit_prior = false en Multinomial, ignorando asi las probabilidades a priori y comportandose similar a ComplementNB. Por otra parte, para ambos se establecieron los mismos parámetros del vectorizador sin modificar ngram_range asi que es posible que esto tambien establezca el limite que pueden lograr ambos modelos.

### Transponer la matriz documento-término.
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

In [245]:
# Matriz termino-documento
X_train_T = X_train.transpose()

# Extraer el vocabulario
vocabulario_array = tfidfvect.get_feature_names_out() # Lista de palabras ordenadas como estan las filas de la matriz traspuesta
vocabulario_dict = tfidfvect.vocabulary_              # Diccionario con palabra: índice de fila en la matriz traspuesta

In [246]:
# Función para buscar similitud de palabras
def get_similar_words(word, vocab_dict, vocab_array, matrix_T, top_n=5):

    # Verifica que la palabra exista
    if word not in vocab_dict:
        return [], []
        
    # Índice de la palabra
    idx = vocab_dict[word]
    # Vector de la palabra (apariciones en todos los documentos)
    word_vector = matrix_T[idx]
    
    # Similitud contra todas las demás palabras
    sim_scores = cosine_similarity(word_vector, matrix_T)[0]
    
    # Obtiene los top_n + 1 índices ordenados
    top_indexes = sim_scores.argsort()[-(top_n + 1):][::-1]
    
    # Elimina la palabra misma (índice 0)
    top_indexes = top_indexes[1:]
    
    # Extrae scores y palabras
    top_scores = sim_scores[top_indexes]
    top_words = [vocab_array[i] for i in top_indexes]
    
    return top_words, top_scores

In [247]:
# Palabras a analizar
selected_words = [
    'galaxy', 'car', 'motherboard', 'jesus', 'gun'
]

for word in selected_words:
    print(f"\n{'='*50}")
    print(f"Palabra: '{word.upper()}'")
    print(f"{'='*50}")
    
    sim_words, sim_scores = get_similar_words(word, vocabulario_dict, vocabulario_array, X_train_T, top_n=5)
    
    if not sim_words:
        print(f"La palabra '{word}' fue filtrada por el vectorizador (stop-word o min_df).")
        continue
        
    results_list = []
    
    for j, (sim_word, sim_score) in enumerate(zip(sim_words, sim_scores), start=1):
        results_list.append({
            'Orden': j,
            'Palabra Similar': sim_word,
            'Similitud Coseno': round(sim_score, 4)
        })
        
    df_results = pd.DataFrame(results_list)
    
    # Mostramos la tabla sin el índice por defecto
    display(df_results.style.hide(axis="index"))


Palabra: 'GALAXY'


Orden,Palabra Similar,Similitud Coseno
1,visix,0.549100
2,uimx,0.546500
3,milky,0.511200
4,galactic,0.469300
5,favourable,0.432400



Palabra: 'CAR'


Orden,Palabra Similar,Similitud Coseno
1,cars,0.179700
2,criterium,0.177000
3,civic,0.174800
4,owner,0.168900
5,dealer,0.168100



Palabra: 'MOTHERBOARD'


Orden,Palabra Similar,Similitud Coseno
1,ami,0.244000
2,micronics,0.231700
3,cyric,0.225900
4,myslef,0.225900
5,dlc,0.225900



Palabra: 'JESUS'


Orden,Palabra Similar,Similitud Coseno
1,christ,0.303900
2,god,0.268800
3,kingdom,0.213000
4,mat,0.196700
5,bible,0.195100



Palabra: 'GUN'


Orden,Palabra Similar,Similitud Coseno
1,guns,0.358200
2,crime,0.244100
3,handgun,0.239100
4,homicides,0.233100
5,firearms,0.232800


#### Análisis de resultados

1. **Galaxy**: Los resultados muestran relacion.**Galactico** y **Lactea** son obvios. Luego **Visix** resulta ser la empresa que desarrolló la interfaz gráfica del juego galaxy y **Uim/x** es otra empresa de desarrollo de interfaces. **Favorable** no parece estar muy relacionada a priori, podrían ser menciones a reseñas favorables del juego o de estas empresas.
2. **Car**: Mayormente relacionadas, **cars**, **civic**, **owner**, **dealer**. **Criterium** es menos evidente.
3. **Motherboard**: Todos fabricantes de motherboards o chips de las mismas en esa época.
4. **Jesus**: Todas palabras relacionadas con religion como **cristo**, **dios**, **reino**, **biblia** a excepcion de **mat** que es un poco dificil a que puede referirse (el evangelio de Mateo? la fima de un usuario muy activo de los foros?).
5. **Gun**: Todas palabras evidentemente relacionadas: crimen, armas de fuego, de mano, homicidios.